# 02 — Build the FAISS Job Index & Semantic Search

1. Load the job CSV (`config.JOBS_CSV`) with pandas. Combine `jobtitle + skills + jobdescription` into one text per job.
2. Embed those job texts and build a **FAISS** index. Save it to `config.JOBS_INDEX_DIR` so the app loads it instead of rebuilding.
3. Embed a candidate profile and run a **top-N** similarity search — return the closest jobs with scores.
4. Sanity-check: do the top jobs actually match the profile? Move the load/query code into `src/search/job_search.py`.

Start with a few thousand jobs while developing so embedding is fast and cheap.

In [3]:
# ============================================================
# SMART HIRE - JOB EMBEDDINGS AND FAISS INDEX
# ============================================================

import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import faiss


# ============================================================
# 1. FIND PROJECT ROOT
# ============================================================

current = Path.cwd()

for folder in [current] + list(current.parents):

    if (
        (folder / "src").is_dir()
        and (folder / "data").is_dir()
        and (folder / "notebooks").is_dir()
    ):
        PROJECT_ROOT = folder
        break

else:
    raise FileNotFoundError(
        "Could not find SmartHire project root."
    )


if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT)
    )


print("Project root:")
print(PROJECT_ROOT)


# ============================================================
# 2. IMPORT PROJECT MODULES
# ============================================================

from src import config

from src.search.embed import (
    embed_text,
    embed_texts
)


# ============================================================
# 3. LOAD JOB CSV
# ============================================================

job_file = config.JOBS_CSV

if not job_file.exists():

    raise FileNotFoundError(
        f"Job CSV not found:\n{job_file}"
    )


jobs_df = pd.read_csv(
    job_file,
    low_memory=False
)


print()
print("=" * 60)
print("JOB DATA LOADED")
print("=" * 60)

print(
    "Number of jobs:",
    len(jobs_df)
)

print(
    "Columns:",
    list(jobs_df.columns)
)


# ============================================================
# 4. CHECK REQUIRED COLUMNS
# ============================================================

required_columns = {
    "jobtitle",
    "jobdescription"
}

missing_columns = (
    required_columns -
    set(jobs_df.columns)
)

if missing_columns:

    raise ValueError(
        "Missing required columns: "
        + str(missing_columns)
    )


# ============================================================
# 5. PREPARE JOB DATA
# ============================================================

jobs_df = jobs_df.fillna("")


jobs_df["jobtitle"] = (
    jobs_df["jobtitle"]
    .astype(str)
)


jobs_df["jobdescription"] = (
    jobs_df["jobdescription"]
    .astype(str)
)


# ------------------------------------------------------------
# Skills column
# ------------------------------------------------------------

if "skills" not in jobs_df.columns:

    jobs_df["skills"] = ""


jobs_df["skills"] = (
    jobs_df["skills"]
    .astype(str)
)


# ============================================================
# 6. CREATE JOB TEXT
# ============================================================

jobs_df["job_text"] = (
    "Job Title: "
    + jobs_df["jobtitle"]
    + "\nSkills: "
    + jobs_df["skills"]
    + "\nJob Description: "
    + jobs_df["jobdescription"]
)


job_texts = (
    jobs_df["job_text"]
    .tolist()
)


print()
print(
    "Number of job texts:",
    len(job_texts)
)


# ============================================================
# 7. CREATE EMBEDDINGS
# ============================================================

print()
print("=" * 60)
print("CREATING JOB EMBEDDINGS")
print("=" * 60)

print(
    "Embedding model:",
    config.EMBED_MODEL
)

print(
    "Embedding dimension:",
    config.EMBED_DIM
)


job_embeddings = embed_texts(
    job_texts,
    batch_size=32
)


job_embeddings = np.asarray(
    job_embeddings,
    dtype=np.float32
)


# ============================================================
# 8. VERIFY EMBEDDINGS
# ============================================================

expected_shape = (
    len(jobs_df),
    config.EMBED_DIM
)


print()
print(
    "Embedding shape:",
    job_embeddings.shape
)


if job_embeddings.shape != expected_shape:

    raise ValueError(
        f"Expected embedding shape "
        f"{expected_shape}, "
        f"but received "
        f"{job_embeddings.shape}"
    )


print(
    "Embedding dimension:",
    job_embeddings.shape[1]
)


# ============================================================
# 9. SAVE RAW EMBEDDINGS
# ============================================================

config.EMBEDDINGS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


embeddings_file = (
    config.EMBEDDINGS_DIR /
    "job_embeddings_bge_small.npy"
)


np.save(
    embeddings_file,
    job_embeddings
)


print()
print(
    "Raw embeddings saved to:"
)

print(
    embeddings_file
)


# ============================================================
# 10. CREATE FAISS INDEX
# ============================================================

print()
print("=" * 60)
print("CREATING FAISS INDEX")
print("=" * 60)


index = faiss.IndexFlatIP(
    config.EMBED_DIM
)


index.add(
    job_embeddings
)


# ============================================================
# 11. VERIFY FAISS INDEX
# ============================================================

print(
    "FAISS index created."
)

print(
    "Vectors stored in FAISS:",
    index.ntotal
)


if index.ntotal != len(jobs_df):

    raise ValueError(
        "FAISS vector count does not "
        "match number of jobs."
    )


if index.d != config.EMBED_DIM:

    raise ValueError(
        f"FAISS dimension {index.d} "
        f"does not match expected "
        f"dimension {config.EMBED_DIM}"
    )


# ============================================================
# 12. CREATE JOBS FAISS DIRECTORY
# ============================================================

config.JOBS_INDEX_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 13. SAVE FAISS INDEX
# ============================================================

faiss_file = (
    config.JOBS_INDEX_DIR /
    "jobs.faiss"
)


faiss.write_index(
    index,
    str(faiss_file)
)


print()
print(
    "FAISS index saved to:"
)

print(
    faiss_file
)


# ============================================================
# 14. CREATE JOB METADATA
# ============================================================

metadata = []


for i, row in jobs_df.iterrows():

    metadata.append(
        {
            "index": int(i),

            "jobid": str(
                row.get(
                    "jobid",
                    ""
                )
            ),

            "jobtitle": str(
                row.get(
                    "jobtitle",
                    ""
                )
            ),

            "skills": str(
                row.get(
                    "skills",
                    ""
                )
            ),

            "jobdescription": str(
                row.get(
                    "jobdescription",
                    ""
                )
            ),

            "job_text": str(
                row.get(
                    "job_text",
                    ""
                )
            )
        }
    )


# ============================================================
# 15. SAVE METADATA AS JSON
# ============================================================

metadata_file = (
    config.JOBS_INDEX_DIR /
    "jobs.json"
)


with open(
    metadata_file,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        metadata,
        file,
        indent=2,
        ensure_ascii=False
    )


print()
print(
    "Job metadata saved to:"
)

print(
    metadata_file
)


# ============================================================
# 16. VERIFY SAVED FILES
# ============================================================

print()
print("=" * 60)
print("VERIFYING SAVED FILES")
print("=" * 60)


if not faiss_file.exists():

    raise FileNotFoundError(
        "jobs.faiss was not created."
    )


if not metadata_file.exists():

    raise FileNotFoundError(
        "jobs.json was not created."
    )


print(
    "✅ jobs.faiss exists"
)

print(
    "✅ jobs.json exists"
)


# ============================================================
# 17. LOAD THE SAVED INDEX AGAIN
# ============================================================

test_index = faiss.read_index(
    str(faiss_file)
)


with open(
    metadata_file,
    "r",
    encoding="utf-8"
) as file:

    test_metadata = json.load(file)


print()
print(
    "Reloaded FAISS vectors:",
    test_index.ntotal
)

print(
    "Reloaded metadata records:",
    len(test_metadata)
)


if test_index.ntotal != len(
    test_metadata
):

    raise ValueError(
        "Reloaded FAISS count and "
        "metadata count do not match."
    )


# ============================================================
# 18. TEST JOB SEARCH
# ============================================================

print()
print("=" * 60)
print("TESTING JOB SEARCH")
print("=" * 60)


test_query = (
    "Python machine learning "
    "data science"
)


print(
    "Test query:",
    test_query
)


query_embedding = embed_text(
    test_query
)


query_embedding = np.asarray(
    query_embedding,
    dtype=np.float32
).reshape(
    1,
    -1
)


if query_embedding.shape[1] != config.EMBED_DIM:

    raise ValueError(
        f"Query embedding dimension "
        f"{query_embedding.shape[1]} "
        f"does not match expected "
        f"{config.EMBED_DIM}"
    )


print(
    "Query embedding shape:",
    query_embedding.shape
)


# ============================================================
# 19. SEARCH TOP 5 JOBS
# ============================================================

top_k = min(
    config.TOP_N_JOBS,
    test_index.ntotal
)


scores, indices = (
    test_index.search(
        query_embedding,
        top_k
    )
)


print()
print(
    f"Top {top_k} matching jobs:"
)


for rank, (
    score,
    idx
) in enumerate(
    zip(
        scores[0],
        indices[0]
    ),
    start=1
):

    if idx < 0:
        continue


    job = test_metadata[
        int(idx)
    ]


    print()
    print(
        f"{rank}. "
        f"{job['jobtitle']}"
    )

    print(
        f"   Job ID: "
        f"{job['jobid']}"
    )

    print(
        f"   Score: "
        f"{score:.4f}"
    )


# ============================================================
# 20. FINAL VALIDATION
# ============================================================

print()
print("=" * 60)
print("JOB EMBEDDINGS CREATED SUCCESSFULLY")
print("=" * 60)

print(
    "Number of jobs:",
    len(jobs_df)
)

print(
    "Number of job texts:",
    len(job_texts)
)

print(
    "Number of embeddings:",
    len(job_embeddings)
)

print(
    "Embedding shape:",
    job_embeddings.shape
)

print(
    "Embedding dimension:",
    config.EMBED_DIM
)

print()
print(
    "FAISS vectors:",
    index.ntotal
)

print(
    "FAISS dimension:",
    index.d
)

print()
print(
    "FAISS index:"
)

print(
    faiss_file
)

print()
print(
    "Job metadata:"
)

print(
    metadata_file
)

print()
print(
    "Raw embeddings:"
)

print(
    embeddings_file
)

print()
print("=" * 60)
print("READY FOR EVALUATION")
print("=" * 60)

Project root:
c:\Users\jeesh\Smart_hire_Gen_AI_June

JOB DATA LOADED
Number of jobs: 22000
Columns: ['company', 'education', 'experience', 'industry', 'jobdescription', 'jobid', 'joblocation_address', 'jobtitle', 'numberofpositions', 'payrate', 'postdate', 'site_name', 'skills', 'uniq_id']

Number of job texts: 22000

CREATING JOB EMBEDDINGS
Embedding model: BAAI/bge-small-en-v1.5
Embedding dimension: 384


Batches: 100%|██████████| 688/688 [46:08<00:00,  4.02s/it]  



Embedding shape: (22000, 384)
Embedding dimension: 384


AttributeError: module 'src.config' has no attribute 'EMBEDDINGS_DIR'

In [4]:
from src import config

print(config.EMBEDDINGS_DIR)

AttributeError: module 'src.config' has no attribute 'EMBEDDINGS_DIR'

In [5]:
from src import config

print(config.EMBEDDINGS_DIR)


AttributeError: module 'src.config' has no attribute 'EMBEDDINGS_DIR'

In [6]:
from pathlib import Path
import sys
import json
import numpy as np
import faiss

# ------------------------------------------------------------
# 1. FIND PROJECT ROOT
# ------------------------------------------------------------

current = Path.cwd()

for folder in [current] + list(current.parents):
    if (
        (folder / "src").is_dir()
        and (folder / "data").is_dir()
        and (folder / "notebooks").is_dir()
    ):
        PROJECT_ROOT = folder
        break
else:
    raise FileNotFoundError("SmartHire project root not found.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


# ------------------------------------------------------------
# 2. IMPORT EMBEDDING FUNCTION
# ------------------------------------------------------------

from src.search.embed import embed_texts


# ------------------------------------------------------------
# 3. SET PATHS
# ------------------------------------------------------------

NOTES_DIR = PROJECT_ROOT / "data" / "career_notes"
OUTPUT_DIR = PROJECT_ROOT / "vectorstore" / "notes_faiss"

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Project:", PROJECT_ROOT)
print("Career notes:", NOTES_DIR)
print("Output:", OUTPUT_DIR)


# ------------------------------------------------------------
# 4. FIND ALL CAREER NOTE FILES
# ------------------------------------------------------------

files = [
    path
    for path in NOTES_DIR.rglob("*")
    if path.is_file()
]

print("\nFiles found:", len(files))

for file in files:
    print("-", file.relative_to(NOTES_DIR))


# ------------------------------------------------------------
# 5. READ FILES
# ------------------------------------------------------------

def read_file(path):

    extension = path.suffix.lower()

    if extension in [".txt", ".md", ".markdown"]:
        return path.read_text(
            encoding="utf-8",
            errors="ignore"
        )

    elif extension == ".pdf":
        from pypdf import PdfReader

        reader = PdfReader(str(path))

        return "\n".join(
            page.extract_text() or ""
            for page in reader.pages
        )

    elif extension == ".docx":
        from docx import Document

        document = Document(str(path))

        return "\n".join(
            paragraph.text
            for paragraph in document.paragraphs
            if paragraph.text.strip()
        )

    elif extension == ".json":
        return path.read_text(
            encoding="utf-8",
            errors="ignore"
        )

    return ""


# ------------------------------------------------------------
# 6. LOAD DOCUMENTS
# ------------------------------------------------------------

documents = []

for file in files:

    try:
        text = read_file(file).strip()

        if text:

            documents.append({
                "source": str(
                    file.relative_to(NOTES_DIR)
                ),
                "text": text
            })

    except Exception as error:

        print(
            "Could not read:",
            file.name,
            "|",
            error
        )


if not documents:
    raise ValueError(
        "No readable career-note files were found."
    )

print("\nDocuments loaded:", len(documents))


# ------------------------------------------------------------
# 7. CREATE CHUNKS
# ------------------------------------------------------------

chunks = []

CHUNK_SIZE = 800
CHUNK_OVERLAP = 150

for document in documents:

    text = document["text"]

    start = 0

    while start < len(text):

        end = start + CHUNK_SIZE

        chunk = text[start:end].strip()

        if chunk:

            chunks.append({
                "source": document["source"],
                "text": chunk
            })

        if end >= len(text):
            break

        start = end - CHUNK_OVERLAP


if not chunks:
    raise ValueError(
        "No chunks were created."
    )

print("Chunks created:", len(chunks))


# ------------------------------------------------------------
# 8. CREATE EMBEDDINGS
# ------------------------------------------------------------

texts = [
    chunk["text"]
    for chunk in chunks
]

print("\nCreating embeddings...")

embeddings = embed_texts(
    texts,
    batch_size=32
)

embeddings = np.asarray(
    embeddings,
    dtype=np.float32
)

print(
    "Embedding shape:",
    embeddings.shape
)


# ------------------------------------------------------------
# 9. CHECK EMBEDDING DIMENSION
# ------------------------------------------------------------

if embeddings.shape[1] != 384:

    raise ValueError(
        f"Expected 384 dimensions, "
        f"but got {embeddings.shape[1]}"
    )


# ------------------------------------------------------------
# 10. CREATE FAISS INDEX
# ------------------------------------------------------------

index = faiss.IndexFlatIP(384)

index.add(embeddings)

print(
    "FAISS vectors:",
    index.ntotal
)


# ------------------------------------------------------------
# 11. CREATE METADATA
# ------------------------------------------------------------

metadata = []

for i, chunk in enumerate(chunks):

    metadata.append({
        "index": i,
        "source": chunk["source"],
        "text": chunk["text"]
    })


# ------------------------------------------------------------
# 12. SAVE FAISS INDEX
# ------------------------------------------------------------

faiss_path = OUTPUT_DIR / "notes.faiss"

faiss.write_index(
    index,
    str(faiss_path)
)


# ------------------------------------------------------------
# 13. SAVE METADATA
# ------------------------------------------------------------

json_path = OUTPUT_DIR / "notes.json"

with open(
    json_path,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        metadata,
        file,
        ensure_ascii=False,
        indent=2
    )


# ------------------------------------------------------------
# 14. VERIFY FILES
# ------------------------------------------------------------

if not faiss_path.exists():
    raise FileNotFoundError(
        "notes.faiss was not created."
    )

if not json_path.exists():
    raise FileNotFoundError(
        "notes.json was not created."
    )


# ------------------------------------------------------------
# 15. RELOAD AND VERIFY
# ------------------------------------------------------------

loaded_index = faiss.read_index(
    str(faiss_path)
)

with open(
    json_path,
    "r",
    encoding="utf-8"
) as file:

    loaded_metadata = json.load(file)


if loaded_index.ntotal != len(loaded_metadata):
    raise ValueError(
        "FAISS and metadata counts do not match."
    )

if loaded_index.d != 384:
    raise ValueError(
        "FAISS dimension is not 384."
    )


# ------------------------------------------------------------
# 16. TEST RETRIEVAL
# ------------------------------------------------------------

test_question = "How can I prepare for a data analyst career?"

query_embedding = embed_texts(
    [test_question]
)

query_embedding = np.asarray(
    query_embedding,
    dtype=np.float32
)

scores, indices = loaded_index.search(
    query_embedding,
    3
)

print("\nTest question:")
print(test_question)

print("\nTop results:")

for rank, (score, idx) in enumerate(
    zip(scores[0], indices[0]),
    start=1
):

    print("\n-----------------------------")
    print("Rank:", rank)
    print("Score:", float(score))
    print(
        "Source:",
        loaded_metadata[idx]["source"]
    )
    print(
        loaded_metadata[idx]["text"][:500]
    )


# ------------------------------------------------------------
# 17. FINAL RESULT
# ------------------------------------------------------------

print("\n")
print("=" * 60)
print("SUCCESS")
print("=" * 60)

print("Documents:", len(documents))
print("Chunks:", len(chunks))
print("Embeddings:", embeddings.shape)
print("FAISS vectors:", loaded_index.ntotal)

print("\nCreated files:")

print(faiss_path)
print(json_path)

print("\nYour notes FAISS vectorstore is ready.")

Project: c:\Users\jeesh\Smart_hire_Gen_AI_June
Career notes: c:\Users\jeesh\Smart_hire_Gen_AI_June\data\career_notes
Output: c:\Users\jeesh\Smart_hire_Gen_AI_June\vectorstore\notes_faiss

Files found: 12
- data_analyst_roadmap.md
- resume_writing_tips.md
- carrer_guides\AI_Engineer_Roadmap_2025.md
- carrer_guides\Backend_Developer_Roadmap.md
- carrer_guides\Career_Roadmap_for_Freshers.md
- carrer_guides\Data_Analyst_Role_Guide.md
- carrer_guides\Frontend_Developer_Roadmap.md
- carrer_guides\Full_Stack_Developer_Role_Guide.md
- carrer_guides\GenAI_Developer_Roadmap.md
- carrer_guides\Interview_Preparation_Guide.md
- carrer_guides\ML_Engineer_Roadmap.md
- carrer_guides\Resume_and_Portfolio_Guide.md

Documents loaded: 12
Chunks created: 83

Creating embeddings...


Batches: 100%|██████████| 3/3 [00:02<00:00,  1.11it/s]


Embedding shape: (83, 384)
FAISS vectors: 83


Batches: 100%|██████████| 1/1 [00:00<00:00, 84.66it/s]


Test question:
How can I prepare for a data analyst career?

Top results:

-----------------------------
Rank: 1
Score: 0.8315907716751099
Source: data_analyst_roadmap.md
# How to Become a Data Analyst

A data analyst collects, cleans, and interprets data to help a business make
decisions. It is one of the most common first roles for people moving into data.

## Core skills
- **SQL** — the single most important skill. You must be able to write SELECT
  queries with JOINs, GROUP BY, and filtering.
- **Spreadsheets** — Excel or Google Sheets: pivot tables, lookups, charts.
- **A visualisation tool** — Power BI or Tableau to build dashboards.
- **Statistics basics**

-----------------------------
Rank: 2
Score: 0.8120579719543457
Source: data_analyst_roadmap.md
for plots.

## Switching from another role
If you are coming from a non-data job, the fastest path is:
1. Learn SQL first and practise on real datasets.
2. Rebuild reports you already make by hand into a dashboard.
3. Do two or th

In [9]:
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import faiss


# ============================================================
# 1. FIND PROJECT ROOT
# ============================================================

current = Path.cwd()

for folder in [current] + list(current.parents):
    if (
        (folder / "src").is_dir()
        and (folder / "data").is_dir()
        and (folder / "notebooks").is_dir()
    ):
        PROJECT_ROOT = folder
        break
else:
    raise FileNotFoundError("SmartHire project root not found.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


# ============================================================
# 2. IMPORT
# ============================================================

from src import config
from src.search.embed import embed_text, embed_texts


# ============================================================
# 3. PATHS
# ============================================================

JOB_FILE = PROJECT_ROOT / "data" / "jobs" / "naukri_com-job_sample.csv"

EMBEDDINGS_DIR = PROJECT_ROOT / "data" / "embeddings"
OUTPUT_DIR = PROJECT_ROOT / "vectorstore" / "jobs_faiss"

EMBEDDINGS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# 4. LOAD JOB DATA
# ============================================================

if not JOB_FILE.exists():
    raise FileNotFoundError(
        f"Job CSV not found:\n{JOB_FILE}"
    )

jobs_df = pd.read_csv(
    JOB_FILE,
    low_memory=False
)

print("=" * 60)
print("JOB DATA")
print("=" * 60)

print("Number of jobs:", len(jobs_df))


# ============================================================
# 5. CHECK COLUMNS
# ============================================================

if "jobtitle" not in jobs_df.columns:
    raise ValueError("jobtitle column is missing.")

if "jobdescription" not in jobs_df.columns:
    raise ValueError("jobdescription column is missing.")


# ============================================================
# 6. PREPARE DATA
# ============================================================

jobs_df = jobs_df.fillna("")

if "skills" not in jobs_df.columns:
    jobs_df["skills"] = ""

jobs_df["jobtitle"] = jobs_df["jobtitle"].astype(str)
jobs_df["skills"] = jobs_df["skills"].astype(str)
jobs_df["jobdescription"] = jobs_df["jobdescription"].astype(str)


# ============================================================
# 7. CREATE JOB TEXT
# ============================================================

jobs_df["job_text"] = (
    "Job Title: "
    + jobs_df["jobtitle"]
    + "\nSkills: "
    + jobs_df["skills"]
    + "\nJob Description: "
    + jobs_df["jobdescription"]
)

job_texts = jobs_df["job_text"].tolist()

print("Job texts:", len(job_texts))


# ============================================================
# 8. CHECK JOB COUNT
# ============================================================

if len(jobs_df) != 22000:
    print(
        f"WARNING: Expected 22000 jobs, "
        f"but CSV contains {len(jobs_df)} jobs."
    )
else:
    print("SUCCESS: CSV contains 22000 jobs.")


# ============================================================
# 9. CREATE EMBEDDINGS
# ============================================================

print()
print("=" * 60)
print("CREATING JOB EMBEDDINGS")
print("=" * 60)

print("Model:", config.EMBED_MODEL)
print("Dimension:", config.EMBED_DIM)

job_embeddings = embed_texts(
    job_texts,
    batch_size=512
)

job_embeddings = np.asarray(
    job_embeddings,
    dtype=np.float32
)

print("Embedding shape:", job_embeddings.shape)


# ============================================================
# 10. VERIFY EMBEDDINGS
# ============================================================

expected_shape = (
    len(jobs_df),
    384
)

if job_embeddings.shape != expected_shape:
    raise ValueError(
        f"Wrong embedding shape.\n"
        f"Expected: {expected_shape}\n"
        f"Received: {job_embeddings.shape}"
    )

print("Embedding verification successful.")


# ============================================================
# 11. SAVE RAW EMBEDDINGS
# ============================================================

embeddings_file = (
    EMBEDDINGS_DIR / "job_embeddings_bge_small.npy"
)

np.save(
    embeddings_file,
    job_embeddings
)

print("Raw embeddings saved:")
print(embeddings_file)


# ============================================================
# 12. CREATE FAISS INDEX
# ============================================================

print()
print("=" * 60)
print("CREATING FAISS INDEX")
print("=" * 60)

index = faiss.IndexFlatIP(384)

index.add(job_embeddings)

print("FAISS vectors:", index.ntotal)


# ============================================================
# 13. VERIFY FAISS COUNT
# ============================================================

if index.ntotal != len(jobs_df):
    raise ValueError(
        f"FAISS contains {index.ntotal} vectors "
        f"but dataset contains {len(jobs_df)} jobs."
    )

if index.ntotal != 22000:
    raise ValueError(
        f"Expected 22000 FAISS vectors "
        f"but got {index.ntotal}."
    )

print("SUCCESS: FAISS contains exactly 22000 vectors.")


# ============================================================
# 14. SAVE FAISS
# ============================================================

faiss_file = OUTPUT_DIR / "jobs.faiss"

faiss.write_index(
    index,
    str(faiss_file)
)

print("FAISS saved:")
print(faiss_file)


# ============================================================
# 15. CREATE METADATA
# ============================================================

metadata = []

for i, row in jobs_df.iterrows():

    metadata.append(
        {
            "index": int(i),
            "jobid": str(row.get("jobid", "")),
            "jobtitle": str(row.get("jobtitle", "")),
            "skills": str(row.get("skills", "")),
            "jobdescription": str(
                row.get("jobdescription", "")
            ),
            "job_text": str(
                row.get("job_text", "")
            )
        }
    )


# ============================================================
# 16. SAVE METADATA
# ============================================================

metadata_file = OUTPUT_DIR / "jobs.json"

with open(
    metadata_file,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        metadata,
        file,
        indent=2,
        ensure_ascii=False
    )

print("Metadata saved:")
print(metadata_file)


# ============================================================
# 17. RELOAD AND VERIFY
# ============================================================

test_index = faiss.read_index(
    str(faiss_file)
)

with open(
    metadata_file,
    "r",
    encoding="utf-8"
) as file:

    test_metadata = json.load(file)

print()
print("=" * 60)
print("FINAL VERIFICATION")
print("=" * 60)

print("CSV jobs:", len(jobs_df))
print("Embeddings:", len(job_embeddings))
print("FAISS vectors:", test_index.ntotal)
print("Metadata records:", len(test_metadata))
print("Embedding dimension:", job_embeddings.shape[1])
print("FAISS dimension:", test_index.d)


# ============================================================
# 18. FINAL CHECKS
# ============================================================

if len(jobs_df) != 22000:
    raise ValueError(
        "The CSV does not contain exactly 22000 jobs."
    )

if len(job_embeddings) != 22000:
    raise ValueError(
        "There are not exactly 22000 embeddings."
    )

if test_index.ntotal != 22000:
    raise ValueError(
        "There are not exactly 22000 FAISS vectors."
    )

if len(test_metadata) != 22000:
    raise ValueError(
        "There are not exactly 22000 metadata records."
    )

if test_index.d != 384:
    raise ValueError(
        "FAISS dimension is not 384."
    )


# ============================================================
# 19. TEST SEARCH
# ============================================================

query = "Python machine learning data science"

query_embedding = embed_text(query)

query_embedding = np.asarray(
    query_embedding,
    dtype=np.float32
).reshape(1, -1)

scores, indices = test_index.search(
    query_embedding,
    5
)

print()
print("=" * 60)
print("TEST SEARCH")
print("=" * 60)

for rank, (score, idx) in enumerate(
    zip(scores[0], indices[0]),
    start=1
):

    job = test_metadata[int(idx)]

    print()
    print(
        f"{rank}. {job['jobtitle']}"
    )

    print(
        "Job ID:",
        job["jobid"]
    )

    print(
        "Score:",
        round(float(score), 4)
    )


# ============================================================
# 20. SUCCESS
# ============================================================

print()
print("=" * 60)
print("SUCCESS")
print("=" * 60)

print("22000 jobs processed")
print("22000 embeddings created")
print("22000 FAISS vectors created")
print("22000 metadata records created")

print()
print("Created:")
print(faiss_file)
print(metadata_file)
print(embeddings_file)

print()
print("READY FOR EVALUATION")

JOB DATA
Number of jobs: 22000
Job texts: 22000
SUCCESS: CSV contains 22000 jobs.

CREATING JOB EMBEDDINGS
Model: BAAI/bge-small-en-v1.5
Dimension: 384


Batches: 100%|██████████| 43/43 [2:46:16<00:00, 232.02s/it]    


Embedding shape: (22000, 384)
Embedding verification successful.
Raw embeddings saved:
c:\Users\jeesh\Smart_hire_Gen_AI_June\data\embeddings\job_embeddings_bge_small.npy

CREATING FAISS INDEX
FAISS vectors: 22000
SUCCESS: FAISS contains exactly 22000 vectors.
FAISS saved:
c:\Users\jeesh\Smart_hire_Gen_AI_June\vectorstore\jobs_faiss\jobs.faiss
Metadata saved:
c:\Users\jeesh\Smart_hire_Gen_AI_June\vectorstore\jobs_faiss\jobs.json

FINAL VERIFICATION
CSV jobs: 22000
Embeddings: 22000
FAISS vectors: 22000
Metadata records: 22000
Embedding dimension: 384
FAISS dimension: 384

TEST SEARCH

1. Data Scientist - Machine Learning
Job ID: 290916004268
Score: 0.7816

2. Machine Learning Scientist
Job ID: 180716900576
Score: 0.779

3. Data Scientist (machine Learning)
Job ID: 280316000026
Score: 0.7672

4. Data Scientist-Machine Learning
Job ID: 250515500956
Score: 0.7642

5. Data Scientist - Machine Learning/nlp
Job ID: 50816900280
Score: 0.7638

SUCCESS
22000 jobs processed
22000 embeddings creat